In [0]:
dbutils.fs.mount(
    source='wasbs://sales@rkdataengsa.blob.core.windows.net',
    mount_point='/mnt/sales',
    extra_configs={'fs.azure.account.key.rkdataengsa.blob.core.windows.net':'STORAGE_ACCESS_KEY'}
)

In [0]:
ordersDf=spark.read.csv('/mnt/sales/landing/orders.csv', inferSchema=True, header=True)
display(ordersDf)

In [0]:
errorFlg=False
ordersCount=ordersDf.count()
print(ordersCount)

distinctOrdersCount=ordersDf.select('order_id').distinct().count()
print(distinctOrdersCount)

if ordersCount != distinctOrdersCount:
    errorFlg=True
if errorFlg:
    dbutils.fs.mv('/mnt/sales/landing/orders.csv','/mnt/sales/discarded')
    dbutils.notebook.exit('{"errorFlg":"true","errorMsg":"Orderid is repeated"}')  
ordersDf.createOrReplaceTempView("orders")

In [0]:
dbServer='rk-data-eng-sqlserver'
dbPort='1433'
dbName='rk-sql-db'
dbUser='rk-sql-user'
dbPassword='sql-password'
databricksScope='salesprojectscope'

In [0]:
connectionUrl='jdbc:sqlserver://{}.database.windows.net:{};database={};user={};'.format(dbServer, dbPort, dbName, dbUser) dbPassword= dbutils.secrets.get(scope=databricksScope, key='sql-password')
connectionProperties={
    'password':dbPassword,
    'driver':'com.microsoft.sqlserver.jdbc.SQLServerDriver'
}

In [0]:
validStatusDf=spark.read.jdbc(url=connectionUrl, table='dbo.valid_order_status', properties=connectionProperties)

In [0]:
display(validStatusDf)

In [0]:
validStatusDf.createOrReplaceTempView("valid_status")

In [0]:
invalidRowsDf=spark.sql("select * from orders where order_status not in (select * from valid_status)")

In [0]:
display(invalidRowsDf)

In [0]:
if invalidRowsDf.count()>0:
    errorFlg=True
if errorFlg:
    dbutils.fs.mv('/mnt/sales/landing/orders.csv','/mnt/sales/discarded')
    dbutils.notebook.exit('{"errorFlg":"true","errorMsg":"Invalid order status found"}')
else:
    dbutils.fs.mv('/mnt/sales/landing/orders.csv','/mnt/sales/staging')
    dbutils.notebook.exit('{"errorFlg":"false","errorMsg":"All good"}')